<a href="https://colab.research.google.com/github/Stubberson/project-collection/blob/main/OSM_tagging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OSM Map Matching
This notebook automates the process for matching Maptionnaire survey responses with OSM.

The method follows these steps:
1. **Data Ingestion**
  * Read CSVs containing coordinate data into Pandas dataframes.
    * Each CSV contains all the mapped features and their coordinates for a survey.
  * Convert to GeoPandas.
2. **Query the Overpass API**
  * Define the Overpass API query.
  * Query OSM with the survey features.
    * Use good API etiquette.
    * Here a sample size of 100 features is used, because querying is quite slow.
3. **OSM Tags and Matching**
  * After the querying, count the most frequent OSM tags per survey.
  * Use the 5 most frequently occurring tags as a filter for matching.
  * Calculate counts of matched OSM tags and distances to the elements with those tags.
4. **Results**
  * Aggregate element specific results for the whole survey.
5. **Convert to CSV**
  * Convert the results geodataframes into CSVs

## 1. Data Ingestion
Retrieve the needed reponse data and convert into GeoPandas dataframe. Convert multiple CSV files into GeoPandas GeoDataFrames, creating the geometry column from the `response_array` column which contains coordinate strings representing Points, Lines, or Areas.

First, extract an uploaded zip-file onto the Colab workspace.

In [ ]:
import zipfile
import os

zip_file = 'coordinate_data.zip'
# Extract the files onto the default working folder
extract_dir = '.'

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Extracted contents of {zip_file}")

Extracted contents of coordinate_data.zip


Then convert into GeoDataFrames.

In [ ]:
import geopandas as gpd
import glob
from shapely.geometry import Point, LineString, Polygon
import json
import pandas as pd

def parse_coordinates_to_geometry(row):
    """
    Parses a coordinate array string from a DataFrame row and creates a Shapely geometry.

    Arg:
        row: A row from a Pandas DataFrame.
    Return:
        A Shapely geometry (Point, LineString, or Polygon) or None if parsing fails.
    """
    coord_array_str = row['response_array']
    element_type = row['elementType']

    try:
        # Load the JSON array from the string
        coords_list = json.loads(coord_array_str)

        # Use a list comprehension for concise and efficient parsing
        coordinates = []
        for item in coords_list:
            try:
                # Split and convert to float, swapping to (lon, lat) order
                lat, lon = map(float, item.split(','))  # map() executes a specified function for each item in an iterable
                coordinates.append((lon, lat))
            except (ValueError, IndexError):
                print(f"Skipping invalid coordinate pair: {item}")
                continue

        if not coordinates:
            return None

        # Create geometry based on element type
        if element_type == 'geoPoint' and len(coordinates) == 1:
            return Point(coordinates[0])
        elif element_type == 'geoLine' and len(coordinates) >= 2:
            return LineString(coordinates)
        elif element_type == 'geoArea' and len(coordinates) >= 3:
            # Close the polygon if it isn't already
            if coordinates[0] != coordinates[-1]:
                coordinates.append(coordinates[0])
            return Polygon(coordinates)
        else:
            return None

    except (json.JSONDecodeError, TypeError, KeyError) as e:
        print(f"Error processing row with element type {element_type}: {e}")
        return None

# --- Main script logic ---

csv_files = glob.glob('coordinate_data/*.csv')

# Create a dictionary to store the GeoDataFrames
geodataframes = {}

# Loop through each CSV file
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file)

        # Apply the single, pre-defined function to create the geometry column
        df['geometry'] = df.apply(parse_coordinates_to_geometry, axis=1)

        # Drop rows where geometry could not be created and convert to GeoDataFrame
        gdf = gpd.GeoDataFrame(df.dropna(subset=['geometry']), geometry='geometry')

        # Check if the GeoDataFrame is empty after cleaning
        if not gdf.empty:
            geodataframes[csv_file] = gdf
            #print(f"Successfully created GeoDataFrame for {csv_file} with {len(gdf)} features.")
        else:
            print(f"No valid geometries found in {csv_file}. Skipping.")

    except (KeyError, FileNotFoundError) as e:
        print(f"Critical error processing {csv_file}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred with {csv_file}: {e}")

print("GeoDataFrames created.")

GeoDataFrames created.


## 2. Query the Overpass API

Formulate and execute queries to the Overpass API to retrieve relevant OSM data based on the geometries in the ingested GeoDataFrames.

Iterate through the geodataframes, construct Overpass queries based on the geometry of each row, execute the queries, and store the results. Use the `requests` library to interact with the Overpass API.

For each response:

`if element_type == 'geoPoint':`
* Query all elements with a tag within a 10 meter radius

`elif element_type == 'geoLine':`
* Query all the intersecting elements with a tag

`elif element_type == 'geoArea':`
* Query all the intersecting and enclosed elements with a tag

In [ ]:
import requests
import time

def build_overpass_query(geometry):
    """
    Builds an Overpass QL query string based on a shapely geometry object.

    Arg:
        geometry: A shapely geometry (Point, LineString, or Polygon).
    Return:
        A string containing the Overpass QL query.
    """

    geom_type = geometry.geom_type

    if geom_type == 'Point':
        # For points, query a 10m radius around them
        lat, lon = geometry.y, geometry.x
        query = f"""
            [out:json];
                (
                    nw(around: 10,{lat},{lon})[~"."~".*"];
                );
            out tags geom qt;
        """
        return query

    elif geom_type == 'LineString':
        coords_list = " ".join([f"{lat}, {lon}{',' if i < len(geometry.coords) - 1 else ''}" for i, (lon, lat) in enumerate(geometry.coords)])
        query = f"""
            [out:json];
                (
                    nw(around: 0, {coords_list})[~"."~".*"];
                );
            out tags geom qt;
        """
        return query

    elif geom_type == 'Polygon':
        # For polygons, query the polygon's interior
        coords_str = " ".join([f"{lat} {lon}" for lon, lat in geometry.exterior.coords])
        query = f"""
            [out:json];
                (
                    nw(poly: "{coords_str}")[~"."~".*"];
                );
            out tags geom qt;
        """
        return query

    else:
        return None

# --- Main script logic ---
# LIMIT (10 000 queries / day):
    # The German (main) instance:    "https://overpass-api.de/api/interpreter"
# NO LIMIT
    # The Austrian instance:         "https://overpass.private.coffee/api/interpreter"
    # The Russian instance:          "https://maps.mail.ru/osm/tools/overpass/api/interpreter"
overpass_url = "https://overpass.private.coffee/api/interpreter"
overpass_results = {}

# Use a session object for potentially better performance
session = requests.Session()

for file_name, gdf in geodataframes.items():
    overpass_results[file_name] = []

    # Querying 100 features per survey with 216 surveys takes ≈5h to query
    sample_size = 100

    # Take a sample only if the number of features is greater than the sample size, otherwise use all features
    if len(gdf) > sample_size:
        sampled_gdf = gdf.sample(n=sample_size, random_state=42)
    else:
        sampled_gdf = gdf

    # Use enumerate to get both the list position and the DataFrame index/row
    for index, row in sampled_gdf.iterrows():
        geometry = row['geometry']
        element_type = row['elementType'] # Get the element type

        if not geometry:
            print(f"Skipping row {index} in {file_name} due to invalid geometry.")
            continue

        # Build the query
        query = build_overpass_query(geometry)

        if not query:
            print(f"Skipping row {index} in {file_name} due to unhandled geometry type: {geometry.geom_type}")
            continue

        try:
            response = session.post(overpass_url, data={'data': query})
            response.raise_for_status()

            # Include original data in the results
            result_data = response.json()
            result_data["original_data"] = {
                "geometry": geometry.wkb,
                "elementType": element_type,
                "response_array": row.get('response_array')
            }
            overpass_results[file_name].append(result_data)

            print(f"Successfully queried for row {index} in {file_name}")
            time.sleep(0.1) # Be polite to the API
        except requests.exceptions.RequestException as e:
            print(f"Error querying for row {index} in {file_name}: {e}")
            overpass_results[file_name].append({"error": str(e), "query": query})
        except Exception as e:
            print(f"An unexpected error occurred for row {index} in {file_name}: {e}")
            overpass_results[file_name].append({"error": str(e)})

print("Finished querying Overpass API.")

Streaming output truncated to the last 5000 lines.
Successfully queried for row 7 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 8 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 9 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 10 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 11 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 12 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 13 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 14 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 15 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 16 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 17 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 18 in coordinate_data/091_6b297yn64bs9.csv
Successfully queried for row 19 in coordinate_data/091_6b297yn64bs9.csv
Successfully que

## 3. Inter-rater Matching

1. Calculate the most frequently tagged OSM tags for each survey.
2. Use the 5 most frequently tagged keys as a filter for matching. Only save the elements that have one (or more) of those tags.
3. Calculate the distances to the matched tags from the response. Average them out. Only Point features' distance is applicable: Lines and Areas intersect the elements, so distance is always 0
    * For calculating distances in meters, it is important to reproject the OSM results into a local CRS.
    * The max distance is always =< 10m, because of the query filter.

In [ ]:
import pyproj
from shapely.ops import transform
import numpy as np
from collections import Counter
from shapely.geometry import Point, LineString, Polygon

def get_utm_crs(geometry):
    """
    Project the geometries onto local CRS for distance calculation.

    Arg:
        geometry: A shapely geometry (Point, LineString, or Polygon).
    Return:
        A pyproj CRS object or None if the geometry is empty.
    """
    if geometry.is_empty:
        return None
    centroid = geometry.centroid
    lon, lat = centroid.x, centroid.y
    utm_zone = int((lon + 180) / 6) + 1
    utm_epsg = 32600 + utm_zone if lat >= 0 else 32700 + utm_zone
    return pyproj.CRS(f"EPSG:{utm_epsg}")

# Helper function to parse OSM geometry
def parse_osm_geometry(element):
    """
    Creates a shapely geometry from an Overpass API element.

    Arg:
        element: A dictionary representing an element from the Overpass API response.
    Return:
        A shapely geometry (Point, LineString, or Polygon) or None if parsing fails.
    """
    try:
        if 'lat' in element and 'lon' in element:
            return Point(element['lon'], element['lat'])
        elif 'geometry' in element and isinstance(element['geometry'], list):
            coords = [(c['lon'], c['lat']) for c in element['geometry'] if 'lon' in c and 'lat' in c]
            if not coords: return None
            # Check if it's a closed polygon
            if len(coords) >= 4 and coords[0] == coords[-1]:
                return Polygon(coords)
            elif len(coords) >= 2:
                return LineString(coords)
    except (KeyError, TypeError) as e:
        print(f"Could not parse geometry for element {element.get('id', 'N/A')}: {e}")
    return None

# --- Main script logic ---
wgs84 = pyproj.CRS("EPSG:4326")
matched_results = {}

for file_name, overpass_data in overpass_results.items():
    matched_results[file_name] = []

    file_tags = Counter()
    keys_to_omit = {'ref', 'source', 'operator', 'note', 'frequency', 'name'}

    for response in filter(None, overpass_data): # filter(None,...) skips any None responses
        for element in response.get('elements', []):
            for key in element.get('tags', {}):
                if not any(key.lower().startswith(prefix) for prefix in keys_to_omit):
                    file_tags[key] += 1

    keys_to_match_with_counts = file_tags.most_common(5)
    keys_to_match = [tag[0] for tag in keys_to_match_with_counts]

    print(f"Determined keys to match for {file_name}: {keys_to_match_with_counts}")

    # Iterate directly over the overpass_data list, which contains the results
    for overpass_response in overpass_data:

        # Ensure the original data is present in the response
        original_data_dict = overpass_response.get("original_data")
        if not original_data_dict:
             matched_results[file_name].append({"match_status": "Original Data Missing"})
             continue

        # Recreate original geometry from the stored GeoJSON
        try:
            original_geometry = gpd.GeoSeries.from_wkb([original_data_dict['geometry']]).iloc[0]
        except Exception as e:
            print(f"Could not recreate original geometry for a row in {file_name}: {e}")
            matched_results[file_name].append({"original_data": original_data_dict, "match_status": "Geometry Recreation Error"})
            continue

        original_element_type = original_data_dict.get('elementType')

        if not (overpass_response and "error" not in overpass_response and original_geometry):
            status = "Query Error or No Results" if "error" in (overpass_response or {}) else "Invalid Original Geometry"
            matched_results[file_name].append({"original_data": original_data_dict, "match_status": status})
            continue

        # --- Projection Step ---
        utm_crs = get_utm_crs(original_geometry)
        if not utm_crs:
            matched_results[file_name].append({"original_data": original_data_dict, "match_status": "Projection Error"})
            continue

        transformer = pyproj.Transformer.from_crs(wgs84, utm_crs, always_xy=True)
        original_geometry_projected = transform(transformer.transform, original_geometry)

        # --- Matching Loop ---
        matched_elements_info = []
        for osm_element in overpass_response.get('elements', []):
            osm_tags = osm_element.get('tags', {})
            osm_geometry = parse_osm_geometry(osm_element)

            if not osm_geometry: continue

            # Check for a matching tag key
            if any(key in osm_tags for key in keys_to_match):
                osm_geometry_projected = transform(transformer.transform, osm_geometry)
                distance = None
                # Only calculate distance for original Points
                if original_geometry.geom_type == 'Point':
                    distance = original_geometry_projected.distance(osm_geometry_projected)

                matched_elements_info.append({
                    "osm_id": osm_element.get('id'),
                    "tags": osm_tags,
                    "distance_m": distance
                })

        # --- Final Aggregation ---
        distances = [info['distance_m'] for info in matched_elements_info if info['distance_m'] is not None]
        avg_distance = np.mean(distances) if distances else None
        sum_distance = sum(distances) if distances else None

        matched_results[file_name].append({
            "original_data": original_data_dict,
            "match_status": "Completed",
            "matched_elements_count": len(matched_elements_info),
            "matched_elements": matched_elements_info,
            "avg_distance_m": avg_distance,
            "sum_distance_m": sum_distance
        })

print("Finished matching process.")

Determined keys to match for coordinate_data/164_8el9aux7hge7.csv: [('highway', 846), ('tiger:cfcc', 127), ('tiger:county', 127), ('oneway', 122), ('tiger:name_base', 121)]
Determined keys to match for coordinate_data/199_9fh33ecl93ho.csv: [('highway', 187), ('surface', 140), ('lit', 129), ('bicycle', 68), ('snowplowing', 60)]
Determined keys to match for coordinate_data/165_8fd7jg2v9dr7.csv: [('highway', 236), ('surface', 146), ('landuse', 71), ('bicycle', 61), ('foot', 59)]
Determined keys to match for coordinate_data/075_4ln4pku9b9u7.csv: [('highway', 179), ('tiger:cfcc', 68), ('tiger:county', 66), ('tiger:name_base', 63), ('oneway', 59)]
Determined keys to match for coordinate_data/039_39zy9hbo8gza.csv: [('highway', 185), ('lanes', 75), ('oneway', 62), ('building', 44), ('surface', 41)]
Determined keys to match for coordinate_data/024_2lf6r7fxu3sa.csv: [('building', 164715), ('highway', 120561), ('addr:housenumber', 112328), ('addr:street', 112263), ('addr:city', 111357)]
Determine

## 4. Aggregate results

`matched_results` are a huge pile of data without any insight. Take some aggregates for further refinement.


In [ ]:
# Initialize dictionary to store aggregated metrics
file_metrics = {}

for file_name, results in matched_results.items():
    tag_key_matches = Counter()
    tag_key_value_matches = Counter()

    metrics = {
        "total_queried_geometries": len(results),
        "geometries_with_match": 0,
        "total_matched_elements": 0,
        "avg_distance_m": 0,
        "element_type_counts": Counter(geodataframes[file_name]['elementType'])
    }

    for result in results:
        # Check if the row had at least one successful match
        matched_elements_count = result.get("matched_elements_count", 0)
        if matched_elements_count > 0:
            metrics["geometries_with_match"] += 1
            metrics["total_matched_elements"] += matched_elements_count

            # --- Tag counting ---
            matched_elements_list = result.get("matched_elements", [])
            for matched_element in matched_elements_list:
                osm_tags = matched_element.get("tags", {})
                for key, value in osm_tags.items():
                    # Omit specified keys during aggregation
                    if not any(key.lower().startswith(prefix) for prefix in keys_to_omit):
                        # Add to counters
                        tag_key_matches[key] += 1
                        tag_key_value_matches[f"{key}:{value}"] += 1

            metrics["avg_distance_m"] = result.get("avg_distance_m", 0)

    # Final calculations
    if metrics["total_queried_geometries"] > 0:
        metrics["percentage_geometries_with_match"] = \
            (metrics["geometries_with_match"] / metrics["total_queried_geometries"]) * 100.0
    else:
        metrics["percentage_geometries_with_match"] = 0

    # Get top 5 of each tag type
    metrics["tag_key_matches"] = tag_key_matches.most_common(5)
    metrics["tag_key_value_matches"] = tag_key_value_matches.most_common(5)


    file_metrics[file_name] = metrics

# Display the calculated metrics
print("\nFile-specific Spatial Accuracy Metrics:")
for file_name, metrics in file_metrics.items():
    print(f"\n{file_name}:")
    print(f"  Queried Geometries: {metrics['total_queried_geometries']}")
    print("  Original Element Type Counts:")
    for element_type, count in metrics['element_type_counts'].items():
        print(f"    {element_type}: {count}")
    print(f"  Geometries with Match: {metrics['geometries_with_match']}")
    print(f"  Percentage Geometries with Match: {metrics['percentage_geometries_with_match']:.2f}%")
    print(f"  Total Matched Elements: {metrics['total_matched_elements']}")
    if metrics['avg_distance_m'] is not None:
        print(f"  Average Distance in Meters (Points only): {metrics['avg_distance_m']:.2f}")
    else:
        print("  Average Distance in Meters (Points only): N/A")
    print("  Top 5 Tag Keys:")
    for tag_key, count in metrics["tag_key_matches"]:
        print(f"    {tag_key}: {count}")
    print("  Top 5 Tag Key-Value Pairs:")
    for tag_key_value, count in metrics["tag_key_value_matches"]:
        print(f"    {tag_key_value}: {count}")


File-specific Spatial Accuracy Metrics:

coordinate_data/164_8el9aux7hge7.csv:
  Queried Geometries: 100
  Original Element Type Counts:
    geoLine: 3292
    geoPoint: 6708
  Geometries with Match: 89
  Percentage Geometries with Match: 89.00%
  Total Matched Elements: 846
  Average Distance in Meters (Points only): 4.58
  Top 5 Tag Keys:
    highway: 846
    tiger:cfcc: 127
    tiger:county: 127
    oneway: 122
    tiger:name_base: 121
  Top 5 Tag Key-Value Pairs:
    highway:footway: 401
    highway:service: 192
    highway:residential: 148
    tiger:county:Horry, SC: 127
    tiger:cfcc:A41: 103

coordinate_data/199_9fh33ecl93ho.csv:
  Queried Geometries: 78
  Original Element Type Counts:
    geoPoint: 78
  Geometries with Match: 59
  Percentage Geometries with Match: 75.64%
  Total Matched Elements: 201
  Average Distance in Meters (Points only): 6.06
  Top 5 Tag Keys:
    highway: 187
    surface: 140
    lit: 129
    bicycle: 68
    snowplowing: 60
  Top 5 Tag Key-Value Pairs:


Explanations for the results:

* *Queried Geometries*: A sample from the original data that was used ot query Overpass API
* *Original Element Type Counts*: Not a sample-wide, but a file/questionnaire-wide count of the different geo features
* *Geometries with Match*: Count of the geometries from *Queried Geometries* that had at least one element with a top 5 tag key.
* *Percentage Geometries with Match*: Same as above, but a percentage.
* *Total Matched Elements*: The count of elements within the matched geometries that have at least one top 5 tag key.
* *Average Distance (Points Only)*: Average distance to the matched elements for point elements (lines only intersect (distance=0) and areas only search within (distance=0))



## 5. Convert to CSV
Convert all the aggregated results into a odwnloadable csv file.

In [ ]:
import csv
# Define the output CSV file name
output_csv_file = 'aggregated_metrics.csv'

# Prepare data for CSV
csv_data = []
# Add header row
header = ["file_name", "total_queried_geometries", "geometries_with_match", "percentage_geometries_with_match",
          "total_matched_elements", "avg_distance_m", "element_type_counts", "tag_key_matches", "tag_key_value_matches"]
csv_data.append(header)

for file_name, metrics in file_metrics.items():
    row = [
        file_name,
        metrics["total_queried_geometries"],
        metrics["geometries_with_match"],
        metrics["percentage_geometries_with_match"],
        metrics["total_matched_elements"],
        metrics["avg_distance_m"],
        str(metrics["element_type_counts"]), # Convert Counter to string
        str(metrics["tag_key_matches"]),     # Convert list of tuples to string
        str(metrics["tag_key_value_matches"]) # Convert list of tuples to string
    ]
    csv_data.append(row)

# Write to CSV
with open(output_csv_file, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(csv_data)

print(f"Aggregated metrics saved to {output_csv_file}")

Aggregated metrics saved to aggregated_metrics.csv
